In [ ]:
# ============================================================
# 🩺 AI SKIN DISEASE ASSISTANT
# FULL VERSION + DIRECT PDF DOWNLOAD
# ============================================================

# ============================================================
# 1. INSTALL PACKAGES
# ============================================================

!pip -q install requests chromadb sentence-transformers pillow transformers torch torchvision reportlab ipywidgets


# ============================================================
# 2. IMPORTS
# ============================================================

import os
import re
import io
import html
import time
import requests
import torch

import chromadb

from PIL import Image, ImageStat, ImageFilter

from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration
)

from sentence_transformers import SentenceTransformer

import ipywidgets as widgets

from IPython.display import (
    display,
    Markdown,
    clear_output
)

# Google Colab PDF download
from google.colab import files

# ReportLab
from reportlab.lib.pagesizes import A4
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Image as RLImage
)
from reportlab.lib.styles import (
    getSampleStyleSheet,
    ParagraphStyle
)
from reportlab.lib.enums import TA_CENTER


# ============================================================
# 3. NEXUS API CONFIG
# ============================================================

print("🔐 Enter your Nexus API Key")

NEXUS_API_KEY = input("Nexus API Key: ").strip()

NEXUS_BASE_URL = "https://nexusapi.navigatelabs.ai"

NEXUS_CHAT_URL = (
    f"{NEXUS_BASE_URL}/chat/completions"
)

NEXUS_MODELS_URL = (
    f"{NEXUS_BASE_URL}/models"
)

MODEL_NAME = "nova-micro"


# ============================================================
# 4. RAG CONFIG
# ============================================================

CHROMA_PATH = "/content/chroma_db"

COLLECTION_NAME = "skin_diseases"

TOP_K = 3

EMBEDDING_MODEL_NAME = (
    "all-MiniLM-L6-v2"
)


# ============================================================
# 5. LOAD BLIP
# ============================================================

print("\n⏳ Loading BLIP image model...")

BLIP_PROCESSOR = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

BLIP_MODEL = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

BLIP_MODEL.eval()

print("✅ BLIP loaded")


# ============================================================
# 6. LOAD EMBEDDING MODEL
# ============================================================

print("\n⏳ Loading embedding model...")

EMBEDDING_MODEL = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("✅ Embedding model loaded")


# ============================================================
# 7. CONNECT CHROMADB
# ============================================================

print("\n⏳ Connecting to ChromaDB...")

chroma_client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

try:

    collection = chroma_client.get_collection(
        name=COLLECTION_NAME
    )

except Exception:

    collection = chroma_client.create_collection(
        name=COLLECTION_NAME
    )

print("✅ ChromaDB ready")


# ============================================================
# 8. TEST NEXUS
# ============================================================

def test_nexus_connection():

    if not NEXUS_API_KEY:

        print("⚠️ Nexus API key not entered.")

        return False

    headers = {
        "Authorization":
            f"Bearer {NEXUS_API_KEY}",
        "Content-Type":
            "application/json"
    }

    try:

        response = requests.get(
            NEXUS_MODELS_URL,
            headers=headers,
            timeout=20
        )

        if response.status_code == 200:

            print(
                "✅ Nexus API connection successful"
            )

            return True

        return False

    except Exception:

        return False


NEXUS_AVAILABLE = test_nexus_connection()


# ============================================================
# 9. FILE UPLOAD HANDLER
# ============================================================

def get_uploaded_file(upload_value):

    if not upload_value:
        return None, None

    # --------------------------------------------------------
    # Dictionary format
    # --------------------------------------------------------

    if isinstance(
        upload_value,
        dict
    ):

        try:

            first_key = next(
                iter(upload_value)
            )

            item = upload_value[
                first_key
            ]

            if isinstance(
                item,
                dict
            ):

                filename = item.get(
                    "name",
                    first_key
                )

                content = item.get(
                    "content"
                )

                if content is not None:

                    return (
                        filename,
                        bytes(content)
                    )

        except Exception:
            pass

    # --------------------------------------------------------
    # List / Tuple format
    # --------------------------------------------------------

    if isinstance(
        upload_value,
        (tuple, list)
    ):

        if len(upload_value) > 0:

            item = upload_value[0]

            if isinstance(
                item,
                dict
            ):

                filename = item.get(
                    "name",
                    "uploaded_image"
                )

                content = item.get(
                    "content"
                )

                if content is not None:

                    return (
                        filename,
                        bytes(content)
                    )

    # --------------------------------------------------------
    # Object format
    # --------------------------------------------------------

    try:

        item = upload_value[0]

        filename = getattr(
            item,
            "name",
            "uploaded_image"
        )

        content = getattr(
            item,
            "content",
            None
        )

        if content is not None:

            return (
                filename,
                bytes(content)
            )

    except Exception:
        pass

    return None, None


# ============================================================
# 10. LOCAL IMAGE ANALYSIS
# ============================================================

def local_image_analysis(image):

    width, height = image.size

    rgb_image = image.convert(
        "RGB"
    )

    stat = ImageStat.Stat(
        rgb_image
    )

    r, g, b = stat.mean

    brightness = (
        r + g + b
    ) / 3

    warm_score = r - b

    grayscale = rgb_image.convert(
        "L"
    )

    contrast = ImageStat.Stat(
        grayscale
    ).stddev[0]

    edges = grayscale.filter(
        ImageFilter.FIND_EDGES
    )

    edge_strength = ImageStat.Stat(
        edges
    ).mean[0]

    # --------------------------------------------------------
    # BLIP caption
    # --------------------------------------------------------

    try:

        inputs = BLIP_PROCESSOR(
            images=rgb_image,
            return_tensors="pt"
        )

        with torch.no_grad():

            output = BLIP_MODEL.generate(
                **inputs,
                max_new_tokens=40
            )

        caption = BLIP_PROCESSOR.decode(
            output[0],
            skip_special_tokens=True
        )

    except Exception:

        caption = (
            "Visual description unavailable."
        )

    return {

        "width": width,

        "height": height,

        "brightness": round(
            brightness,
            2
        ),

        "average_red": round(
            r,
            2
        ),

        "average_green": round(
            g,
            2
        ),

        "average_blue": round(
            b,
            2
        ),

        "warm_score": round(
            warm_score,
            2
        ),

        "contrast": round(
            contrast,
            2
        ),

        "edge_strength": round(
            edge_strength,
            2
        ),

        "caption": caption
    }


# ============================================================
# 11. VISUAL INFORMATION
# ============================================================

def build_visual_information(data):

    return f"""
Image resolution:
{data['width']} x {data['height']} pixels

Average RGB:
R={data['average_red']}
G={data['average_green']}
B={data['average_blue']}

Average brightness:
{data['brightness']}

Contrast:
{data['contrast']}

Edge strength:
{data['edge_strength']}

Warm tone score:
{data['warm_score']}

Local image caption:
{data['caption']}
"""


# ============================================================
# 12. SYMPTOM INFORMATION
# ============================================================

def build_symptom_information():

    return f"""
Itching:
{itching.value}

Pain:
{pain.value}

Duration:
{duration.value}

Spreading:
{spreading.value}

Bleeding:
{bleeding.value}

Discharge:
{discharge.value}

Other symptoms:
{other_symptoms.value}
"""


# ============================================================
# 13. NEXUS AI ANALYSIS
# ============================================================

def nexus_text_analysis(
    visual_information,
    symptom_information
):

    if not NEXUS_API_KEY:

        return None

    prompt = f"""
You are an AI assistant for preliminary
educational skin-image assessment.

IMPORTANT:

You are NOT a doctor.

Do NOT provide a definitive diagnosis.

Do NOT prescribe medication.

Do NOT recommend treatment.

Do NOT claim certainty.

Analyze only the information supplied below.

VISUAL INFORMATION:

{visual_information}

USER-REPORTED SYMPTOMS:

{symptom_information}

Produce a concise structured preliminary assessment.

Use exactly these sections:

## 🩺 Preliminary Visual Assessment

### 1. Visible Observations

Describe visible characteristics only.

### 2. Symptoms or Visual Characteristics

Summarize the supplied symptoms and visual characteristics.

### 3. Possible Considerations

Mention broad possible categories only.

Use cautious wording such as:

- may be consistent with
- could have several causes
- cannot be determined from image alone

Do NOT give a definitive diagnosis.

### 4. Warning Signs

Mention general warning signs that would justify medical evaluation.

Do NOT claim the user has those warning signs unless they were explicitly reported.

### 5. When to Consult a Dermatologist

Explain when professional evaluation may be appropriate.

Finish with:

Medical disclaimer:
This is an educational preliminary visual assessment and is NOT a medical diagnosis.
It should not replace professional medical advice.
"""

    headers = {

        "Authorization":
            f"Bearer {NEXUS_API_KEY}",

        "Content-Type":
            "application/json"
    }

    payload = {

        "model":
            MODEL_NAME,

        "messages": [

            {
                "role":
                    "system",

                "content":
                    (
                        "You are a careful medical-information "
                        "assistant. Never provide a definitive diagnosis."
                    )
            },

            {
                "role":
                    "user",

                "content":
                    prompt
            }
        ],

        "temperature":
            0.2,

        "max_tokens":
            1200
    }

    try:

        response = requests.post(
            NEXUS_CHAT_URL,
            headers=headers,
            json=payload,
            timeout=90
        )

        if response.status_code != 200:

            return None

        result = response.json()

        choices = result.get(
            "choices",
            []
        )

        if not choices:

            return None

        message = choices[0].get(
            "message",
            {}
        )

        content = message.get(
            "content",
            ""
        )

        if not content:

            return None

        return content.strip()

    except Exception:

        return None


# ============================================================
# 14. LOCAL FALLBACK
# ============================================================

def create_local_fallback(
    visual_information,
    symptom_information
):

    return f"""
## 🩺 Preliminary Visual Assessment

### 1. Visible Observations

The uploaded image was successfully processed locally.

Basic visual characteristics were extracted from the image.

The available image information alone is not sufficient to determine a specific skin condition.

### 2. Symptoms or Visual Characteristics

The following user-provided symptom information was recorded:

{symptom_information}

### 3. Possible Considerations

The visible skin changes may have several possible causes.

A specific condition cannot be reliably identified from this image alone.

### 4. Warning Signs

Medical evaluation is especially important if the area is rapidly worsening, severely painful, repeatedly bleeding, producing pus or discharge, associated with fever, or failing to heal.

### 5. When to Consult a Dermatologist

Consider professional evaluation if the skin change persists, worsens, spreads, becomes painful, bleeds, develops discharge, or causes significant concern.

Medical disclaimer:
This is an educational preliminary visual assessment and is NOT a medical diagnosis.
It should not replace professional medical advice.
"""


# ============================================================
# 15. CLEAN RESPONSE
# ============================================================

def clean_response(response):

    if not response:

        return create_local_fallback(
            "",
            ""
        )

    response = re.sub(
        r'(?im)^#+\s*🩺\s*Preliminary Visual Assessment\s*$',
        '',
        response
    )

    response = response.strip()

    return (
        "## 🩺 Preliminary Visual Assessment\n\n"
        + response
    )


# ============================================================
# 16. RAG QUERY
# ============================================================

def create_rag_query(text):

    if not text:

        return ""

    text = re.sub(
        r'(?is)medical disclaimer:.*',
        '',
        text
    )

    text = re.sub(
        r'#{1,6}\s*',
        '',
        text
    )

    text = re.sub(
        r'\s+',
        ' ',
        text
    )

    return text.strip()[:3000]


# ============================================================
# 17. RED FLAG DETECTION
# ============================================================

def detect_red_flags_from_symptoms():

    flags = []

    # Bleeding
    if bleeding.value == "Yes":

        flags.append(
            "⚠️ Bleeding was explicitly reported."
        )

    # Severe pain
    if pain.value == "Severe":

        flags.append(
            "⚠️ Severe pain was explicitly reported."
        )

    # Spreading
    if spreading.value == "Yes":

        flags.append(
            "⚠️ Spreading or worsening of the skin changes was reported."
        )

    # Discharge
    if discharge.value == "Yes":

        flags.append(
            "⚠️ Discharge was explicitly reported."
        )

    # Additional symptoms
    other = (
        other_symptoms.value or ""
    ).lower()

    if "pus" in other:

        flags.append(
            "⚠️ Pus was mentioned in the additional symptoms."
        )

    if "fever" in other:

        flags.append(
            "⚠️ Fever was mentioned in the additional symptoms."
        )

    if "swelling" in other:

        flags.append(
            "⚠️ Swelling was mentioned in the additional symptoms."
        )

    if (
        "non-healing" in other
        or "non healing" in other
    ):

        flags.append(
            "⚠️ A non-healing skin lesion was mentioned."
        )

    return flags


# ============================================================
# 18. RAG RETRIEVAL
# ============================================================

def retrieve_rag_context(query):

    if not query:

        return None

    try:

        query_embedding = (
            EMBEDDING_MODEL
            .encode(query)
            .tolist()
        )

        results = collection.query(
            query_embeddings=[
                query_embedding
            ],
            n_results=TOP_K
        )

        return results

    except Exception:

        return None


# ============================================================
# 19. DISPLAY RAG
# ============================================================

def display_rag_results(results):

    display(
        Markdown(
            "## 📚 RAG Knowledge Support"
        )
    )

    if not results:

        display(
            Markdown(
                "No RAG knowledge records were available."
            )
        )

        return

    documents = results.get(
        "documents",
        [[]]
    )[0]

    metadatas = results.get(
        "metadatas",
        [[]]
    )[0]

    distances = results.get(
        "distances",
        [[]]
    )[0]

    if not documents:

        display(
            Markdown(
                "No matching knowledge records were found."
            )
        )

        return

    for i, document in enumerate(
        documents
    ):

        metadata = (
            metadatas[i]
            if i < len(metadatas)
            else {}
        )

        distance = (
            distances[i]
            if i < len(distances)
            else None
        )

        condition = metadata.get(
            "condition",
            "Unknown"
        )

        display(
            Markdown(
                f"""
### Evidence {i + 1}

**Condition:** {condition}

**Semantic distance:** {distance}

**Document:**

{document}
"""
            )
        )

    display(
        Markdown(
            """
> ℹ️ Semantic distance is a retrieval similarity
> measure. It is NOT a diagnosis confidence score.
"""
        )
    )


# ============================================================
# 20. PDF CREATION
# ============================================================

def create_pdf_report(
    image,
    filename,
    symptoms,
    visual_data,
    assessment,
    red_flags,
    rag_results
):

    # --------------------------------------------------------
    # UNIQUE PDF NAME
    # --------------------------------------------------------

    timestamp = int(
        time.time()
    )

    safe_name = re.sub(
        r'[^a-zA-Z0-9_-]',
        '_',
        os.path.splitext(filename)[0]
    )

    pdf_path = (
        f"/content/"
        f"skin_assessment_{safe_name}_{timestamp}.pdf"
    )

    # --------------------------------------------------------
    # PDF DOCUMENT
    # --------------------------------------------------------

    doc = SimpleDocTemplate(
        pdf_path,
        pagesize=A4,
        rightMargin=40,
        leftMargin=40,
        topMargin=40,
        bottomMargin=40
    )

    styles = getSampleStyleSheet()

    title_style = ParagraphStyle(
        "TitleCustom",
        parent=styles["Title"],
        alignment=TA_CENTER,
        fontSize=18,
        spaceAfter=15
    )

    heading_style = ParagraphStyle(
        "HeadingCustom",
        parent=styles["Heading2"],
        fontSize=13,
        spaceBefore=10,
        spaceAfter=6
    )

    body_style = ParagraphStyle(
        "BodyCustom",
        parent=styles["BodyText"],
        fontSize=9,
        leading=13,
        spaceAfter=6
    )

    story = []

    # --------------------------------------------------------
    # TITLE
    # --------------------------------------------------------

    story.append(
        Paragraph(
            "🩺 AI Skin Disease Assistant",
            title_style
        )
    )

    story.append(
        Paragraph(
            "Preliminary Visual Skin Assessment Report",
            body_style
        )
    )

    story.append(
        Spacer(1, 10)
    )

    # --------------------------------------------------------
    # IMAGE
    # --------------------------------------------------------

    try:

        report_image_path = (
            "/content/report_uploaded_image.jpg"
        )

        image.convert(
            "RGB"
        ).save(
            report_image_path,
            quality=85
        )

        report_img = RLImage(
            report_image_path,
            width=250,
            height=250
        )

        story.append(
            report_img
        )

        story.append(
            Spacer(1, 10)
        )

    except Exception:

        pass

    # --------------------------------------------------------
    # USER SYMPTOMS
    # --------------------------------------------------------

    story.append(
        Paragraph(
            "User Symptoms",
            heading_style
        )
    )

    symptom_html = html.escape(
        symptoms
    ).replace(
        "\n",
        "<br/>"
    )

    story.append(
        Paragraph(
            symptom_html,
            body_style
        )
    )

    # --------------------------------------------------------
    # LOCAL IMAGE ANALYSIS
    # --------------------------------------------------------

    story.append(
        Paragraph(
            "Local Image Analysis",
            heading_style
        )
    )

    visual_html = f"""
Resolution:
{visual_data['width']} x {visual_data['height']}<br/>

Brightness:
{visual_data['brightness']}<br/>

Contrast:
{visual_data['contrast']}<br/>

Edge strength:
{visual_data['edge_strength']}<br/>

Warm tone score:
{visual_data['warm_score']}<br/>

Image caption:
{html.escape(
    visual_data['caption']
)}
"""

    story.append(
        Paragraph(
            visual_html,
            body_style
        )
    )

    # --------------------------------------------------------
    # AI ASSESSMENT
    # --------------------------------------------------------

    story.append(
        Paragraph(
            "Preliminary Assessment",
            heading_style
        )
    )

    assessment_html = html.escape(
        assessment
    ).replace(
        "\n",
        "<br/>"
    )

    story.append(
        Paragraph(
            assessment_html,
            body_style
        )
    )

    # --------------------------------------------------------
    # RED FLAGS
    # --------------------------------------------------------

    story.append(
        Paragraph(
            "Red-Flag Detection",
            heading_style
        )
    )

    if red_flags:

        for flag in red_flags:

            story.append(
                Paragraph(
                    html.escape(flag),
                    body_style
                )
            )

        story.append(
            Paragraph(
                "Professional medical evaluation may be appropriate, "
                "especially if these symptoms are severe, worsening, "
                "or persistent.",
                body_style
            )
        )

    else:

        story.append(
            Paragraph(
                "No red flags were explicitly reported in the supplied "
                "symptom inputs. This does not rule out a medical condition.",
                body_style
            )
        )

    # --------------------------------------------------------
    # RAG
    # --------------------------------------------------------

    story.append(
        Paragraph(
            "RAG Knowledge Support",
            heading_style
        )
    )

    if rag_results:

        documents = rag_results.get(
            "documents",
            [[]]
        )[0]

        metadatas = rag_results.get(
            "metadatas",
            [[]]
        )[0]

        distances = rag_results.get(
            "distances",
            [[]]
        )[0]

        for i, document in enumerate(
            documents
        ):

            metadata = (
                metadatas[i]
                if i < len(metadatas)
                else {}
            )

            distance = (
                distances[i]
                if i < len(distances)
                else None
            )

            condition = metadata.get(
                "condition",
                "Unknown"
            )

            story.append(
                Paragraph(
                    f"<b>Evidence {i + 1}</b>",
                    body_style
                )
            )

            document_html = html.escape(
                str(document)
            ).replace(
                "\n",
                "<br/>"
            )

            story.append(
                Paragraph(
                    f"""
<b>Condition:</b>
{html.escape(str(condition))}<br/>

<b>Semantic distance:</b>
{distance}<br/>

{document_html}
""",
                    body_style
                )
            )

    # --------------------------------------------------------
    # DISCLAIMER
    # --------------------------------------------------------

    story.append(
        Spacer(1, 10)
    )

    story.append(
        Paragraph(
            """
<b>Medical Disclaimer:</b>
This report is for educational and informational purposes only.
It is NOT a medical diagnosis and should not replace evaluation
by a qualified healthcare professional.
""",
            body_style
        )
    )

    # --------------------------------------------------------
    # BUILD PDF
    # --------------------------------------------------------

    doc.build(
        story
    )

    # --------------------------------------------------------
    # VERIFY FILE
    # --------------------------------------------------------

    if not os.path.exists(
        pdf_path
    ):

        raise FileNotFoundError(
            "PDF file was not created."
        )

    file_size = os.path.getsize(
        pdf_path
    )

    if file_size == 0:

        raise ValueError(
            "PDF file is empty."
        )

    return pdf_path


# ============================================================
# 21. UI
# ============================================================

display(
    Markdown(
        """
# 🩺 AI Skin Disease Assistant

### AI-powered preliminary visual skin assessment with RAG knowledge support

Upload a skin image and provide symptoms for a preliminary educational assessment.

⚠️ **Medical Disclaimer:**
This tool does NOT provide a medical diagnosis and does NOT replace professional medical advice.
"""
    )
)


# ============================================================
# IMAGE UPLOAD
# ============================================================

display(
    Markdown(
        "### 📷 Upload Skin Image"
    )
)

upload = widgets.FileUpload(
    accept="image/*",
    multiple=False
)

display(
    upload
)


# ============================================================
# SYMPTOMS
# ============================================================

display(
    Markdown(
        "### 📝 Symptoms"
    )
)


itching = widgets.Dropdown(
    options=[
        "Not sure",
        "No",
        "Yes",
        "Mild",
        "Moderate",
        "Severe"
    ],
    value="Not sure",
    description="Itching:"
)

pain = widgets.Dropdown(
    options=[
        "Not sure",
        "No",
        "Yes",
        "Mild",
        "Moderate",
        "Severe"
    ],
    value="Not sure",
    description="Pain:"
)

duration = widgets.Dropdown(
    options=[
        "Not sure",
        "Less than 1 week",
        "1–4 weeks",
        "1–3 months",
        "More than 3 months"
    ],
    value="Not sure",
    description="Duration:"
)

spreading = widgets.Dropdown(
    options=[
        "Not sure",
        "No",
        "Yes"
    ],
    value="Not sure",
    description="Spreading:"
)

bleeding = widgets.Dropdown(
    options=[
        "Not sure",
        "No",
        "Yes"
    ],
    value="Not sure",
    description="Bleeding:"
)

discharge = widgets.Dropdown(
    options=[
        "Not sure",
        "No",
        "Yes"
    ],
    value="Not sure",
    description="Discharge:"
)

other_symptoms = widgets.Textarea(
    placeholder=(
        "Example: redness, burning, fever, swelling, "
        "pus, non-healing area..."
    ),
    description="Other:",
    layout=widgets.Layout(
        width="700px",
        height="90px"
    )
)

display(itching)
display(pain)
display(duration)
display(spreading)
display(bleeding)
display(discharge)
display(other_symptoms)


# ============================================================
# ANALYZE BUTTON
# ============================================================

analyze_button = widgets.Button(
    description="🔍 Analyze Skin Image",
    button_style="primary",
    layout=widgets.Layout(
        width="250px",
        height="45px"
    )
)

output = widgets.Output()

display(
    analyze_button
)

display(
    output
)


# ============================================================
# 22. MAIN ANALYSIS
# ============================================================

def run_analysis(button):

    with output:

        clear_output(
            wait=True
        )

        print(
            "⏳ Starting analysis..."
        )

        # ----------------------------------------------------
        # GET IMAGE
        # ----------------------------------------------------

        file_name, file_content = (
            get_uploaded_file(
                upload.value
            )
        )

        if not file_content:

            display(
                Markdown(
                    """
## ❌ Image Upload Required

Please upload a valid skin image before clicking
**Analyze Skin Image**.
"""
                )
            )

            return

        # ----------------------------------------------------
        # OPEN IMAGE
        # ----------------------------------------------------

        try:

            image = Image.open(
                io.BytesIO(
                    file_content
                )
            ).convert("RGB")

        except Exception:

            display(
                Markdown(
                    """
## ❌ Unable to Read Image

The uploaded file could not be opened.

Please upload a valid JPG, JPEG, PNG,
or another supported image.
"""
                )
            )

            return

        # ----------------------------------------------------
        # DISPLAY IMAGE
        # ----------------------------------------------------

        display(
            Markdown(
                f"""
### 📷 Uploaded Image

**File:** `{file_name}`
"""
            )
        )

        display(
            image
        )

        # ----------------------------------------------------
        # SYMPTOMS
        # ----------------------------------------------------

        symptom_information = (
            build_symptom_information()
        )

        # ----------------------------------------------------
        # LOCAL IMAGE ANALYSIS
        # ----------------------------------------------------

        print(
            "\n🔎 Running local image analysis..."
        )

        visual_data = (
            local_image_analysis(
                image
            )
        )

        visual_information = (
            build_visual_information(
                visual_data
            )
        )

        print(
            "✅ Local image analysis completed"
        )

        # ----------------------------------------------------
        # NEXUS
        # ----------------------------------------------------

        print(
            "\n🤖 Running Nexus AI analysis..."
        )

        nexus_result = (
            nexus_text_analysis(
                visual_information,
                symptom_information
            )
        )

        if nexus_result:

            final_report = (
                clean_response(
                    nexus_result
                )
            )

            print(
                "✅ Nexus AI analysis completed"
            )

        else:

            final_report = (
                clean_response(
                    create_local_fallback(
                        visual_information,
                        symptom_information
                    )
                )
            )

            print(
                "✅ Preliminary assessment generated"
            )

        # ----------------------------------------------------
        # DISPLAY ASSESSMENT
        # ----------------------------------------------------

        display(
            Markdown(
                final_report
            )
        )

        # ----------------------------------------------------
        # RED FLAGS
        # ----------------------------------------------------

        display(
            Markdown(
                "## 🚨 Warning / Red-Flag Detection"
            )
        )

        red_flags = (
            detect_red_flags_from_symptoms()
        )

        if red_flags:

            for flag in red_flags:

                display(
                    Markdown(
                        flag
                    )
                )

            display(
                Markdown(
                    """
**👉 Professional medical evaluation may be appropriate,
especially if these symptoms are severe, worsening,
or persistent.**
"""
                )
            )

        else:

            display(
                Markdown(
                    """
✅ No red flags were explicitly reported
in the supplied symptom inputs.

This does **not** rule out a medical condition.
"""
                )
            )

        # ----------------------------------------------------
        # RAG QUERY
        # ----------------------------------------------------

        print(
            "\n📚 Creating RAG query..."
        )

        rag_query = (
            create_rag_query(
                final_report
            )
        )

        rag_query = (
            rag_query
            + "\n"
            + symptom_information
        )

        # ----------------------------------------------------
        # RAG SEARCH
        # ----------------------------------------------------

        print(
            "🔎 Searching knowledge base..."
        )

        rag_results = (
            retrieve_rag_context(
                rag_query
            )
        )

        display_rag_results(
            rag_results
        )

        # ----------------------------------------------------
        # CREATE PDF
        # ----------------------------------------------------

        print(
            "\n📄 Creating PDF report..."
        )

        try:

            pdf_path = (
                create_pdf_report(
                    image=image,
                    filename=file_name,
                    symptoms=symptom_information,
                    visual_data=visual_data,
                    assessment=final_report,
                    red_flags=red_flags,
                    rag_results=rag_results
                )
            )

            print(
                "✅ PDF report created successfully"
            )

        except Exception as e:

            display(
                Markdown(
                    f"""
## ❌ PDF Creation Failed

The PDF could not be created.

Error:
`{str(e)}`
"""
                )
            )

            return

        # ----------------------------------------------------
        # VERIFY
        # ----------------------------------------------------

        pdf_size = os.path.getsize(
            pdf_path
        )

        print(
            f"📄 PDF size: {pdf_size / 1024:.2f} KB"
        )

        # ----------------------------------------------------
        # DIRECT DOWNLOAD
        # ----------------------------------------------------

        display(
            Markdown(
                """
# 📄 Final Report

✅ PDF report is ready.

### ⬇️ Downloading your PDF now...
"""
            )
        )

        # Direct Google Colab browser download
        files.download(
            pdf_path
        )

        # ----------------------------------------------------
        # FINAL MESSAGE
        # ----------------------------------------------------

        display(
            Markdown(
                """
---

## ✅ Analysis Completed

The PDF report contains:

- 📷 Uploaded image
- 📝 User symptoms
- 🔎 Local image analysis
- 🤖 AI preliminary assessment
- 🚨 Red-flag detection
- 📚 RAG knowledge support
- ⚠️ Medical disclaimer

**Important:** This is an educational preliminary assessment,
not a medical diagnosis.
"""
            )
        )


# ============================================================
# CONNECT BUTTON
# ============================================================

analyze_button.on_click(
    run_analysis
)


# ============================================================
# READY
# ============================================================

print("\n" + "=" * 55)

print(
    "✅ AI SKIN DISEASE ASSISTANT READY"
)

print(
    "📷 Upload image"
)

print(
    "📝 Enter symptoms"
)

print(
    "🔍 Click Analyze Skin Image"
)

print(
    "📄 PDF will be downloaded automatically"
)

print("=" * 55)

🔐 Enter your Nexus API Key
Nexus API Key: sk-bvMYU0RNcm5kJdLHZTLaoA

⏳ Loading BLIP image model...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

✅ BLIP loaded

⏳ Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Embedding model loaded

⏳ Connecting to ChromaDB...
✅ ChromaDB ready
✅ Nexus API connection successful



# 🩺 AI Skin Disease Assistant

### AI-powered preliminary visual skin assessment with RAG knowledge support

Upload a skin image and provide symptoms for a preliminary educational assessment.

⚠️ **Medical Disclaimer:**  
This tool does NOT provide a medical diagnosis and does NOT replace professional medical advice.


... (9 outputs left out)

Textarea(value='', description='Other:', layout=Layout(height='90px', width='700px'), placeholder='Example: re…

Button(button_style='primary', description='🔍 Analyze Skin Image', layout=Layout(height='45px', width='250px')…

Output()


✅ AI SKIN DISEASE ASSISTANT READY
📷 Upload image
📝 Enter symptoms
🔍 Click Analyze Skin Image
📄 PDF will be downloaded automatically
